**-Setup**

In [0]:
from pyspark.sql.functions import (
    col, current_timestamp, lit, row_number, to_date, coalesce
)
from pyspark.sql.window import Window

# Widgets
dbutils.widgets.text("storage_account", "retailprojec1data")
dbutils.widgets.text("bronze_container", "bronze")
dbutils.widgets.text("silver_container", "silver")

storage_account = dbutils.widgets.get("storage_account")
bronze_container = dbutils.widgets.get("bronze_container")
silver_container = dbutils.widgets.get("silver_container")

# Paths
bronze_inventory_path = f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net/inventory_delta/"
silver_current_inv_path = f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/current_inventory/"
quarantine_path = f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/_quarantine/current_inventory/"

print(f"Bronze:     {bronze_inventory_path}")
print(f"Silver:     {silver_current_inv_path}")
print(f"Quarantine: {quarantine_path}")

**-Reading Bronze**

In [0]:
bronze_inventory = spark.read.format("delta").load(bronze_inventory_path)
print(f"Bronze inventory rows: {bronze_inventory.count()}")
bronze_inventory.printSchema()
bronze_inventory.show(5, truncate=False)

**-Parse Dates with Quarantine**

In [0]:
from pyspark.sql.functions import (
    col, current_timestamp, lit, row_number, expr, coalesce
)
from pyspark.sql.window import Window

# Parse date with multiple format support — using try_to_date for failure tolerance
inventory_with_date = bronze_inventory.withColumn(
    "snapshot_date_parsed",
    coalesce(
        expr("try_to_date(snapshot_date, 'M/d/yyyy')"),
        expr("try_to_date(snapshot_date, 'MM/dd/yyyy')"),
        expr("try_to_date(snapshot_date, 'yyyy-MM-dd')"),
        expr("try_to_date(snapshot_date)")
    )
)

# Quarantine rows where date parsing failed (all formats returned NULL)
parse_failures = inventory_with_date.filter(col("snapshot_date_parsed").isNull())
failure_count = parse_failures.count()

if failure_count > 0:
    print(f"⚠️ {failure_count} rows failed date parsing — sending to quarantine")
    (parse_failures
        .withColumn("quarantine_reason", lit("snapshot_date_parse_failed"))
        .withColumn("quarantine_ts", current_timestamp())
        .write.format("delta").mode("append").save(quarantine_path))
    print(f"   Quarantined at: {quarantine_path}")
else:
    print("✅ All rows parsed successfully")

# Keep only successfully parsed rows
clean_inventory = inventory_with_date.filter(col("snapshot_date_parsed").isNotNull())
print(f"✅ Clean rows for processing: {clean_inventory.count()}")

**-Finding latest snapshot**

In [0]:
# Window: partition by store_id + sku, order by snapshot_date DESC
window_spec = Window.partitionBy("store_id", "sku").orderBy(col("snapshot_date_parsed").desc())

# Add row_number and filter to latest
latest_inventory = (clean_inventory
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn", "snapshot_date")  # Drop old string date and row_number
    .withColumnRenamed("snapshot_date_parsed", "snapshot_date")
)

print(f"Latest rows per (store, SKU): {latest_inventory.count()}")
latest_inventory.show(10, truncate=False)

**-Adding silver metadata and write**

In [0]:
final = (latest_inventory
    .select(
        "store_id",
        "sku",
        "stock_on_hand",
        "reorder_point",
        "snapshot_date",
    )
    .withColumn("silver_ingestion_ts", current_timestamp())
)

# Write to Silver (overwrite — this is a snapshot table, not appended)
(final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_current_inv_path))

count = spark.read.format("delta").load(silver_current_inv_path).count()
print(f"✅ Silver current_inventory written with {count} rows")

**-Verify**

In [0]:
silver_inv = spark.read.format("delta").load(silver_current_inv_path)

print("=" * 60)
print("current_inventory Summary")
print("=" * 60)
print(f"Total rows:          {silver_inv.count()}")
print(f"Unique stores:       {silver_inv.select('store_id').distinct().count()}")
print(f"Unique SKUs:         {silver_inv.select('sku').distinct().count()}")
print(f"Max snapshot_date:   {silver_inv.agg({'snapshot_date': 'max'}).collect()[0][0]}")
print(f"Min snapshot_date:   {silver_inv.agg({'snapshot_date': 'min'}).collect()[0][0]}")

print("\nSample rows:")
silver_inv.show(10, truncate=False)

print("\nLow inventory items (qty < reorder_point):")
silver_inv.filter(col("stock_on_hand") < col("reorder_point")).show(10, truncate=False)

print("\nSchema:")
silver_inv.printSchema()

**-Exit**

In [0]:
total = silver_inv.count()
low_stock = silver_inv.filter(col("stock_on_hand") < col("reorder_point")).count()
dbutils.notebook.exit(f"total_{total}_low_stock_{low_stock}")